In [1]:
import timeit
import cv2
import numba
from numba import cuda
import numpy as np

In [11]:
@cuda.jit
def gpu_median_filter(src_img, dst_img) -> None:
    """
    CUDA-ядро для применения медианного фильтра.

    Args:
        src_img: входное изображение
        dst_img: выходное изображение
    """
    tid = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    h = src_img.shape[0] - 2
    w = src_img.shape[1] - 2

    while tid < h * w:
        window = cuda.local.array(9, dtype=numba.float64)

        r = tid // w
        c = tid % w

        for i in range(9):
            window[i] = src_img[r + i // 3, c + i % 3]

        for i in range(8):
            for j in range(8 - i):
                if window[j] > window[j + 1]:
                    tmp = window[j]
                    window[j] = window[j + 1]
                    window[j + 1] = tmp

        dst_img[r, c] = window[4]

        tid += cuda.blockDim.x * cuda.gridDim.x

In [12]:
def cpu_median_filter(src_img: np.ndarray) -> np.ndarray:
    """
    CPU-реализация медианного фильтра.

    Args:
        src_img: входное изображение

    Returns:
        результат фильтрации
    """
    h = src_img.shape[0] - 2
    w = src_img.shape[1] - 2

    result = np.zeros((h, w))

    for r in range(h):
        for c in range(w):
            block = src_img[r:r+3, c:c+3].reshape(-1)
            block.sort()
            result[r, c] = block[4]

    return result

In [13]:
def run_gpu_filter(src_img: np.ndarray, dst_img: np.ndarray) -> np.ndarray:
    """
    Запуск медианного фильтра на GPU.

    Args:
        src_img: входное изображение
        dst_img: массив под результат

    Returns:
        результат фильтрации
    """
    d_src = cuda.to_device(src_img)
    d_dst = cuda.device_array_like(dst_img)

    threads = cuda.get_current_device().WARP_SIZE
    blocks = 1024

    gpu_median_filter[blocks, threads](d_src, d_dst)

    return d_dst.copy_to_host()

In [ ]:
noise_level = 0.3

orig = cv2.imread('main.jpg', cv2.IMREAD_GRAYSCALE)

noisy = np.where(
    np.random.rand(*orig.shape) < noise_level,
    np.random.randint(0, 255, orig.shape),
    orig
)

cv2.imwrite('noisy_image.jpg', noisy)

True

In [15]:
iterations = 12

padded = np.pad(noisy, 1, mode="symmetric")
out = np.zeros(noisy.shape)

gpu_out = run_gpu_filter(padded, out)
cpu_out = cpu_median_filter(padded)

print(np.allclose(cpu_out, gpu_out))

cv2.imwrite('filtered_gpu.jpg', gpu_out)
cv2.imwrite('filtered_cpu.jpg', cpu_out)

True


True

In [16]:
gpu_t = timeit.timeit(
    lambda: run_gpu_filter(padded, out),
    number=iterations
) / iterations

cpu_t = timeit.timeit(
    lambda: cpu_median_filter(padded),
    number=iterations
) / iterations

print(f"GPU Execution Time = {gpu_t}")
print(f"CPU Execution Time = {cpu_t}")
print(f"Speedup = {cpu_t / gpu_t}")

GPU Execution Time = 0.0046307702500030246
CPU Execution Time = 1.6658249211666696
Speedup = 359.7295549623913
